 ### 1. Установка готовых библиотек

In [ ]:
from huggingface_hub import login
login(token="your_token_here")

In [ ]:
# !pip install torch transformers accelerate bitsandbytes datasets tqdm

import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


 ### 2. Загрузка модели с 8-битным квантованием

In [ ]:
MODEL_ID = "google/gemma-3-4b-it"

quant_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print("Модель успешно загружена в память.")



In [ ]:
def generate(
    prompt: str, temperature: float = 0.1, max_tokens: int = 128
) -> str:
    """Генерация с использованием chat-шаблона модели."""
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    # Переносим все тензоры на устройство модели
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,  # распаковываем input_ids и attention_mask
            max_new_tokens=max_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Длина входного промпта берётся из input_ids
    input_len = inputs["input_ids"].shape[1]
    response_tokens = outputs[0][input_len:]
    return tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

 ### 3. Методы детекции галлюцинаций (Zero-shot, One-shot, Self-Verification)

In [ ]:
# 3.1. Zero-shot детекция
def detect_zero_shot(passage: str, question: str, answer: str) -> bool:
    prompt = f"""You are an evaluation judge. Your task is to determine whether the provided answer contains any hallucinated or non-factual information that is NOT supported by the text.

Passage:
{passage}

Question:
{question}

Answer:
{answer}

Does the answer contain hallucinated or unverified information?
Respond strictly with "Yes" or "No".

Verdict:"""
    response = generate(prompt, temperature=0.0, max_tokens=10)
    return "YES" in response.upper()



In [ ]:
# 3.2. One-shot детекция
def detect_one_shot(passage: str, question: str, answer: str) -> bool:
    prompt = f"""You are an evaluation judge. Your task is to determine whether the provided answer contains any hallucinated or non-factual information that is NOT supported by the text.

Example:
Passage: The Eiffel Tower in Paris was completed in 1889 for the World's Fair.
Question: Where is the Eiffel Tower located and who built it?
Answer: The Eiffel Tower is in Paris and was designed by Leonardo da Vinci.
Verdict: Yes (Explanation: Leonardo da Vinci is not supported by the passage)

Passage:
{passage}

Question:
{question}

Answer:
{answer}

Does the answer contain hallucinated or unverified information?
Respond strictly with "Yes" or "No".

Verdict:"""
    response = generate(prompt, temperature=0.0, max_tokens=10)
    return "YES" in response.upper()



In [ ]:
# 3.3. Self-Verification цикл (Анализ утверждений -> Проверка вердикта)
def detect_self_verified(passage: str, question: str, answer: str) -> bool:
    # Шаг 1: Извлечение фактов и первичное сопоставление
    analysis_prompt = f"""You are a rigorous factual verifier.
Check if every single claim in the candidate answer is directly supported by the passage.

Passage:
{passage}

Question:
{question}

Candidate Answer:
{answer}

Task:
1. List the key factual claims made in the answer.
2. Check if each claim is verified by the passage or fabricated.
3. State your preliminary judgment: [HALLUCINATED] or [FACTUAL]."""

    analysis = generate(analysis_prompt, temperature=0.1, max_tokens=200)

    # Шаг 2: Самопроверка (Self-reflection) и финальный бинарный вердикт
    verification_prompt = f"""Review your preliminary analysis against the original text.
Ensure you did not make a false positive or overlook subtle unsupported details.

Passage:
{passage}

Answer:
{answer}

Your Analysis:
{analysis}

Final Judgment: Does the answer contain ANY hallucination not grounded in the passage?
Respond strictly with "Yes" or "No".

Verdict:"""

    final_response = generate(
        verification_prompt, temperature=0.0, max_tokens=10
    )
    return "YES" in final_response.upper()



 ### 4. Прогон на сэмпле датасета HaluEval

In [ ]:
dataset = load_dataset("flowaicom/HaluEval", split="test")

SAMPLE_COUNT = dataset.num_rows # для теста можно пару десятков прогнать
eval_data = dataset.select(range(SAMPLE_COUNT))

stats = {
    "zero_shot": {"correct": 0, "false_positives": 0, "false_negatives": 0},
    "one_shot": {"correct": 0, "false_positives": 0, "false_negatives": 0},
    "self_verified": {
        "correct": 0,
        "false_positives": 0,
        "false_negatives": 0,
    },
    "total_samples": 0,
    "total_hallucinations": 0,
    "total_factual": 0,
}

# Эталонные показатели Gemma 2 27B
REF_GEMMA_2_27B = {
    "accuracy": 0.814,
    "missed_hallucination_rate": 0.186,  # False Negative Rate
}

print(f"Старт детекции галлюцинаций на {SAMPLE_COUNT} примерах...")

for item in tqdm(eval_data):
    passage = item["passage"]
    question = item["question"]
    answer = item["answer"]
    raw_label = str(item["label"]).lower()

    # True, если в ответе содержится галлюцинация
    is_hallucinated_gt = raw_label in ["hallucinated", "1", "true", "yes"]

    stats["total_samples"] += 1
    if is_hallucinated_gt:
        stats["total_hallucinations"] += 1
    else:
        stats["total_factual"] += 1

    preds = {
        "zero_shot": detect_zero_shot(passage, question, answer),
        "one_shot": detect_one_shot(passage, question, answer),
        "self_verified": detect_self_verified(passage, question, answer),
    }

    for method, pred in preds.items():
        if pred == is_hallucinated_gt:
            stats[method]["correct"] += 1
        elif pred and not is_hallucinated_gt:
            stats[method]["false_positives"] += 1
        elif not pred and is_hallucinated_gt:
            stats[method]["false_negatives"] += 1


 ### 5. Итоговые результаты и сравнение с Gemma 2 27B

In [ ]:
total = stats["total_samples"]
total_hal = max(stats["total_hallucinations"], 1)

print("\n" + "=" * 70)
print(
    f"{'Configuration':<28} | {'Accuracy':<10} | {'Missed Hallucinations (FNR)':<25}"
)
print("-" * 70)

for method in ["zero_shot", "one_shot", "self_verified"]:
    acc = stats[method]["correct"] / total
    fnr = stats[method]["false_negatives"] / total_hal
    display = f"Gemma 3 4B ({method})"
    print(f"{display:<28} | {acc:<10.3f} | {fnr:<25.3f}")

print("-" * 70)
print(
    f"{'Gemma 2 27B (Reference)':<28} | "
    f"{REF_GEMMA_2_27B['accuracy']:<10.3f} | "
    f"{REF_GEMMA_2_27B['missed_hallucination_rate']:<25.3f}"
)
print("=" * 70)